In [19]:
import sys
import os

# Add parent directory to path to import from app
sys.path.insert(0, os.path.abspath('..'))

from app.db.core.market_data_models import *
from app.db.core.user_data_models import *
from app.db.core.prophit_alts_models import *
from app.db.core.db_config import *
from app.utils.decorators.database import with_session, with_transaction
from app.utils.serialize_output import serialize_sqlalchemy_obj
import json
import pandas as pd
from app.core.calculations.core.helpers import build_returns_df_for_dates
from datetime import datetime, timedelta
import requests
import os
from dotenv import load_dotenv
from sklearn.decomposition import PCA

# Load API key
load_dotenv()
api_key = os.getenv("FMP_API_KEY")

In [20]:
end_date = datetime.now()
start_date = end_date - timedelta(days=365*3)

tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA']

equity_returns_df = build_returns_df_for_dates(
    tickers,
    start_date,
    end_date,
    include_dividends=False,
    drop_rows='any'
)

def get_fmp_data(url):
    """Helper function to fetch data from FMP API"""
    separator = '&' if '?' in url else '?'
    full_url = f"{url}{separator}apikey={api_key}"
    try:
        response = requests.get(full_url)
        response.raise_for_status()
        data = response.json()
        return pd.DataFrame(data) if isinstance(data, list) else pd.DataFrame([data])
    except Exception as e:
        print(f"Error: {e}")
        return pd.DataFrame()

def get_treasury_rates():
    # Treasury Yields - get and rename to desired format
    treasury_data = get_fmp_data("https://financialmodelingprep.com/api/v4/treasury")

    # Rename columns: month1 -> 1m, year1 -> 1y, etc.
    column_mapping = {
        'month1': '1m', 'month3': '3m', 'month6': '6m',
        'year1': '1y', 'year2': '2y', 'year3': '3y', 'year5': '5y', 'year7': '7y', 'year10': '10y', 'year20': '20y', 'year30': '30y'
    }
    treasury_data = treasury_data.rename(columns=column_mapping)

    # Select only the desired columns
    treasury_rates = treasury_data[['date', '1m', '3m', '6m', '1y', '2y', '3y', '5y', '7y', '10y', '20y', '30y']]

    return treasury_rates

rates_df = get_treasury_rates()


In [21]:
def calc_spreads(df):
    df = df.copy()
    df['10y_2y'] = df['10y'] - df['2y']
    df['10y_3m'] = df['10y'] - df['3m']
    df['5y_2y']  = df['5y'] - df['2y']
    return df[['date', '10y_2y', '10y_3m', '5y_2y']]


In [22]:
def calc_volatility(df, window=20):
    yields = df.drop(columns=['date'])
    vol = yields.rolling(window).std()
    vol['date'] = df['date']
    return vol


In [23]:

def calc_pca(df, n_components=3):
    yields = df.drop(columns=['date']).dropna()
    pca = PCA(n_components=n_components)
    pcs = pca.fit_transform(yields - yields.mean())
    pc_df = pd.DataFrame(pcs, columns=[f'PC{i+1}' for i in range(n_components)])
    pc_df['date'] = df.loc[yields.index, 'date'].values
    explained = pca.explained_variance_ratio_
    return pc_df, explained


In [24]:
def calc_correlation(df):
    yields = df.drop(columns=['date'])
    changes = yields.diff().dropna()
    return changes.corr()

In [25]:
def calc_daily_changes(df):
    changes = df.drop(columns=['date']).diff()
    changes['date'] = df['date']
    return changes

In [26]:
def calc_inversion_signals(df):
    df = df.copy()
    df['10y_2y'] = df['10y'] - df['2y']
    df['10y_3m'] = df['10y'] - df['3m']
    
    # Boolean flags
    df['inv_10y2y'] = df['10y_2y'] < 0
    df['inv_10y3m'] = df['10y_3m'] < 0
    
    # Persistence (# of inverted days in a row)
    df['inv_10y2y_streak'] = df['inv_10y2y'].groupby((df['inv_10y2y'] != df['inv_10y2y'].shift()).cumsum()).cumsum()
    df['inv_10y3m_streak'] = df['inv_10y3m'].groupby((df['inv_10y3m'] != df['inv_10y3m'].shift()).cumsum()).cumsum()
    
    # Depth
    avg_depth_2y = df.loc[df['10y_2y'] < 0, '10y_2y'].mean()
    avg_depth_3m = df.loc[df['10y_3m'] < 0, '10y_3m'].mean()
    
    return df[['date','10y_2y','10y_3m','inv_10y2y','inv_10y3m','inv_10y2y_streak','inv_10y3m_streak']], {
        "avg_depth_10y2y": avg_depth_2y,
        "avg_depth_10y3m": avg_depth_3m
    }

In [27]:
def calc_term_premium(df):
    short_end = df[['1y','2y']].mean(axis=1)
    long_end  = df[['20y','30y']].mean(axis=1)
    term_premium = long_end - short_end
    return pd.DataFrame({'date': df['date'], 'term_premium': term_premium})

In [28]:
print(calc_term_premium(rates_df))

          date  term_premium
0   2025-10-02         1.090
1   2025-10-01         1.120
2   2025-09-30         1.080
3   2025-09-29         1.060
4   2025-09-26         1.105
..         ...           ...
59  2025-07-10         0.900
60  2025-07-09         0.905
61  2025-07-08         0.940
62  2025-07-07         0.935
63  2025-07-03         0.890

[64 rows x 2 columns]
